# CMDFusion x UAVScenes — 19-Class Pipeline

**Key Pipeline Architecture & Design**:
1. **Classes**: 19 classes total (18 genuine UAVScenes foreground classes + 1 background class).
2. **mIoU Metric**: Background (Class 0) is learned during training, but **strictly excluded** from the foreground mIoU metric.
3. **Model**: CMDFusion (SPVCNN-G 3D backbone + ResNet-50 2D backbone + Bidirectional Cross-Modal Fusion + Knowledge Distillation).
4. **Pipeline**: Identical data pipeline as PMNet — same splits, calibration, label mapping, metrics.
5. **Only the model is replaced** — everything else is the same as `pmnet_uavscenes_pipeline.ipynb`.

**CMDFusion Architecture** (from [ICRA 2024 paper](https://arxiv.org/abs/2307.04091)):
- **3D Branch**: SPVCNN-G (Sparse Point-Voxel Convolution) with multi-scale encoding
- **2D Branch**: ResNet-50 FCN (pretrained on ImageNet)
- **Fusion**: Bidirectional gated fusion (2D→3D and 3D→2D)
- **KD Loss**: Cross-modal knowledge distillation (MSE between 3D-derived and 2D features)

**Dependencies**: `torch`, `torchvision`, `spconv`, `torch_scatter`, `numpy`, `Pillow`, `matplotlib`

## 0. Setup

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# All imports come from THIS directory only.
THIS_DIR = os.path.abspath(os.path.dirname('__file__'))
# If running from the notebook's own directory, THIS_DIR is already correct.
# Otherwise, set it manually:
# THIS_DIR = '/Users/orbit/Desktop/CMDFusion/uav_cmdfusion'
if THIS_DIR not in sys.path:
    sys.path.insert(0, THIS_DIR)
PROJECT_ROOT = os.path.dirname(THIS_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ============================================================
# Hyperparameters & Optimization Flags
# ============================================================
# --- CMDFusion Model Config ---
NUM_CLASSES    = 19
INPUT_DIMS     = 3       # UAVScenes has XYZ only (no intensity)
HIDEN_SIZE     = 128     # Hidden feature dimension for SPVCNN-G and fusion
SCALE_LIST     = [2, 4, 8, 16]  # Multi-scale voxel scales
BACKBONE_2D    = 'resnet50'      # 2D image backbone
PRETRAINED_2D  = True            # Use ImageNet pretrained weights
LAMBDA_SEG2D   = 4.0             # Weight for KD loss
LAMBDA_XM      = 0.05            # Weight for cross-modal loss

# --- Volume Space (adjust based on your UAVScenes coordinate ranges) ---
MIN_VOLUME_SPACE = [-100, -100, -50]
MAX_VOLUME_SPACE = [100, 100, 150]
SPATIAL_SHAPE    = [400, 400, 40]

# --- Image Config ---
IMG_H = 360     # Target image height for feature map correspondence
IMG_W = 480     # Target image width for feature map correspondence
IMG_TARGET_SIZE = [IMG_H, IMG_W]
IMAGE_NORMALIZER = [[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]]  # ImageNet stats

# --- Training Config ---
BATCH_SIZE = 2       # CMDFusion uses more memory (sparse voxels), smaller batch
MAX_EPOCH  = 64
LR         = 0.001   # Adam learning rate
LR_STEP    = 20      # StepLR period
LR_DECAY   = 0.5     # StepLR gamma
USE_AMP    = True     # Mixed precision training
GRAD_CLIP  = 1.0     # Gradient clipping max norm
VAL_EVERY  = 1       # Validate every N epochs

SAVE_DIR    = os.path.join(THIS_DIR, 'checkpoints')
os.makedirs(SAVE_DIR, exist_ok=True)

NUM_WORKERS = 4
PIN_MEMORY  = True

# Set to a checkpoint path to skip training (e.g. for evaluation only):
CKPT_PATH = None

print(f'Config: {NUM_CLASSES} classes, hidden={HIDEN_SIZE}, scales={SCALE_LIST}')
print(f'Volume: {MIN_VOLUME_SPACE} → {MAX_VOLUME_SPACE}, shape={SPATIAL_SHAPE}')
print(f'Image: {IMG_H}x{IMG_W}, backbone={BACKBONE_2D}')
print(f'Training: batch={BATCH_SIZE}, epochs={MAX_EPOCH}, lr={LR}, AMP={USE_AMP}')

## 1. Data Integrity Checks (mandatory)

Builds geo-diverse train/val/test split, verifies label directories, prints frame counts.
**Do not skip — if this cell raises, stop and investigate.**

In [ ]:
from data_utils import (
    map_labels_26_to_19, map_labels_26_to_8, get_calibration, project_lidar_to_image,
    get_train_val_test_split, build_frame_list, compute_class_weights,
    compute_metrics, verify_sequence_labels, detect_vertical_axis,
    lidar_xyz_to_bev_coords, UNVERIFIED_LABEL_SEQUENCES, CLASS_NAMES,
    IGNORE_INDEX, NUM_CLASSES as NC, CAM_LIDAR_DIR, LIDAR_LABEL_DIR,
)
from uav_cmdfusion_dataset import UAVScenesCMDFusionDataset, collate_fn_uav
from model_cmdfusion_uav import CMDFusionUAV, build_cmdfusion_uav

assert NUM_CLASSES == NC, (
    f'NUM_CLASSES mismatch: notebook={NUM_CLASSES}, data_utils={NC}. Fix above.'
)

# 19-class color palette
PALETTE = np.array([
    [0.60, 0.60, 0.60],  # 0  Background
    [0.86, 0.20, 0.20],  # 1  Roof
    [0.72, 0.53, 0.04],  # 2  Dirt Road
    [0.25, 0.25, 0.25],  # 3  Paved Road
    [0.13, 0.55, 0.87],  # 4  River
    [0.00, 0.75, 0.87],  # 5  Pool
    [0.55, 0.27, 0.07],  # 6  Bridge
    [0.90, 0.90, 0.00],  # 7  Container
    [0.40, 0.40, 0.40],  # 8  Airstrip
    [0.99, 0.55, 0.00],  # 9  Traffic Barrier
    [0.20, 0.80, 0.20],  # 10 Green Field
    [0.55, 0.76, 0.34],  # 11 Wild Field
    [0.10, 0.10, 0.44],  # 12 Solar Panel
    [0.80, 0.47, 0.65],  # 13 Umbrella
    [0.70, 0.85, 0.90],  # 14 Transparent Roof
    [0.50, 0.50, 0.50],  # 15 Car Park
    [0.78, 0.76, 0.70],  # 16 Paved Walk
    [0.00, 0.40, 0.80],  # 17 Sedan
    [0.90, 0.30, 0.00],  # 18 Truck
], dtype=np.float32)
IGNORE_COLOR = np.array([0.95, 0.95, 0.95])

print('Excluded sequences (unverified semantic labels):')
for s in sorted(UNVERIFIED_LABEL_SEQUENCES):
    print(f'  - {s}')
print()

train_seqs, val_seqs, test_seqs = get_train_val_test_split(verify=True)
print(f'Train sequences ({len(train_seqs)}): {train_seqs}')
print(f'Val   sequences ({len(val_seqs)}):   {val_seqs}')
print(f'Test  sequences ({len(test_seqs)}):  {test_seqs}')

locations = ['AMtown', 'AMvalley', 'HKairport', 'HKisland']
print()
print('Geographic coverage check:')
for split_name, seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    present = sorted({loc for loc in locations if any(loc in s for s in seqs)})
    missing = sorted(set(locations) - set(present))
    status = 'OK' if not missing else f'MISSING: {missing}'
    print(f'  {split_name:5s}: {present}  -> {status}')

print()
train_frames = build_frame_list(train_seqs)
val_frames   = build_frame_list(val_seqs)
test_frames  = build_frame_list(test_seqs)
print(f'Frames -- Train: {len(train_frames)}, Val: {len(val_frames)}, Test: {len(test_frames)}')

## 2. Visualize BEFORE Training

Shows RGB, GT labels (19-class), BEV full cloud, and BEV FOV-cropped vs full for one frame per split.

In [ ]:
def visualize_frame(seq_name, frame_idx=0):
    lidar_dir = os.path.join(CAM_LIDAR_DIR, seq_name, 'interval5_LIDAR')
    cam_dir   = os.path.join(CAM_LIDAR_DIR, seq_name, 'interval5_CAM')
    label_dir = os.path.join(LIDAR_LABEL_DIR, seq_name, 'interval5_LIDAR_label_id')

    lidar_files = sorted(f for f in os.listdir(lidar_dir) if f.endswith('.npy'))
    fname = lidar_files[min(frame_idx, len(lidar_files)-1)]
    cam_ts = os.path.splitext(fname)[0].split('_')[0].replace('image', '', 1)

    pts_raw = np.load(os.path.join(lidar_dir, fname)).astype(np.float64)
    pts = pts_raw[:, :3]
    labels_26 = np.load(os.path.join(label_dir, fname)).astype(np.int64)
    labels = map_labels_26_to_19(labels_26)
    img = Image.open(os.path.join(cam_dir, f'{cam_ts}.jpg'))

    calib = get_calibration(seq_name)
    u, v, valid_front = project_lidar_to_image(pts, calib)
    img_w, img_h = img.size
    in_fov = valid_front & (u >= 0) & (u < img_w) & (v >= 0) & (v < img_h)

    bev_all = lidar_xyz_to_bev_coords(pts)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(img)
    axes[0].set_title(f'{seq_name}\nRGB Image ({img_w}x{img_h})')
    axes[0].axis('off')

    axes[1].imshow(img)
    sc = axes[1].scatter(
        u[in_fov], v[in_fov],
        c=[PALETTE[l] for l in labels[in_fov]],
        s=1.0, alpha=0.7,
    )
    axes[1].set_title(f'Projected LiDAR on Image\n({in_fov.sum():,} points in FOV)')
    axes[1].set_xlim(0, img_w); axes[1].set_ylim(img_h, 0)
    axes[1].axis('off')

    axes[2].scatter(
        bev_all[:, 0], bev_all[:, 1],
        c=[PALETTE[l] for l in labels],
        s=0.5, alpha=0.6,
    )
    axes[2].set_title(f'BEV Full Cloud (19 classes)\n({len(pts):,} points, scale preserved)')
    axes[2].set_aspect('equal'); axes[2].axis('off')

    axes[3].scatter(bev_all[~in_fov, 0], bev_all[~in_fov, 1],
                    c='tab:gray', s=0.5, alpha=0.3, label='Outside FOV')
    axes[3].scatter(bev_all[in_fov, 0], bev_all[in_fov, 1],
                    c='tab:red', s=0.8, alpha=0.8, label='In Camera FOV')
    axes[3].set_title(f'BEV FOV-Cropped vs Full\n({in_fov.mean()*100:.1f}% inside camera FOV)')
    axes[3].set_aspect('equal'); axes[3].axis('off')
    axes[3].legend(loc='upper right', markerscale=5)

    plt.suptitle(f'{seq_name} -- Frame #{frame_idx}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('Visualizing one sample frame per split (19 classes):')
visualize_frame('interval5_AMtown01', frame_idx=0)
visualize_frame('interval5_AMvalley02', frame_idx=0)
visualize_frame('interval5_HKairport03', frame_idx=0)

## 3. Class Weights

Smooth logarithmic inverse-frequency class weights.
Ensures all classes receive active, stable gradients without exploding rare classes or starving dominant classes.

In [ ]:
print('Scanning training frames for class balance (19 classes)...')
class_weights = compute_class_weights(train_frames, num_classes=NUM_CLASSES, weight_cap=5.0)

print('=' * 60)
print('CLASS WEIGHTS (19 Classes -- Background = class 0)')
print('=' * 60)
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    tag = ' (Background)' if i == 0 else ''
    print(f'  [{i:2d}] {name:20s}: weight = {w:6.3f}{tag}')
print('=' * 60)

## 4. DataLoaders (CMDFusion Format)

Uses `UAVScenesCMDFusionDataset` which returns variable-length point clouds with
`batch_idx`, `img_indices`, and `point2img_index` — the format CMDFusion expects.
Custom `collate_fn_uav()` handles batching variable-length clouds.

In [ ]:
import sys, importlib
if 'uav_cmdfusion_dataset' in sys.modules:
    importlib.reload(sys.modules['uav_cmdfusion_dataset'])
from uav_cmdfusion_dataset import UAVScenesCMDFusionDataset, collate_fn_uav

# Build datasets
train_ds = UAVScenesCMDFusionDataset(
    sequences=train_seqs, img_h=IMG_H, img_w=IMG_W,
    augment=True,
    min_volume_space=MIN_VOLUME_SPACE,
    max_volume_space=MAX_VOLUME_SPACE,
    image_normalizer=IMAGE_NORMALIZER,
)
val_ds = UAVScenesCMDFusionDataset(
    sequences=val_seqs, img_h=IMG_H, img_w=IMG_W,
    augment=False,
    min_volume_space=MIN_VOLUME_SPACE,
    max_volume_space=MAX_VOLUME_SPACE,
    image_normalizer=IMAGE_NORMALIZER,
)
test_ds = UAVScenesCMDFusionDataset(
    sequences=test_seqs, img_h=IMG_H, img_w=IMG_W,
    augment=False,
    min_volume_space=MIN_VOLUME_SPACE,
    max_volume_space=MAX_VOLUME_SPACE,
    image_normalizer=IMAGE_NORMALIZER,
)

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
    'persistent_workers': True if NUM_WORKERS > 0 else False,
    'prefetch_factor': 2 if NUM_WORKERS > 0 else None,
}

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, collate_fn=collate_fn_uav, **loader_kwargs)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          drop_last=False, collate_fn=collate_fn_uav, **loader_kwargs)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          drop_last=False, collate_fn=collate_fn_uav, **loader_kwargs)

print(f'Train: {len(train_loader)} batches/epoch (batch_size={BATCH_SIZE})')
print(f'Val:   {len(val_loader)} batches')
print(f'Test:  {len(test_loader)} batches')

# Quick shape check
batch = next(iter(train_loader))
print('\nBatch contents:')
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f'  {k:20s}: {tuple(v.shape)}  dtype={v.dtype}')
    elif isinstance(v, list) and len(v) > 0:
        if torch.is_tensor(v[0]):
            print(f'  {k:20s}: list of {len(v)} tensors, [0]={tuple(v[0].shape)}')
        elif isinstance(v[0], np.ndarray):
            print(f'  {k:20s}: list of {len(v)} arrays, [0]={v[0].shape}')
        else:
            print(f'  {k:20s}: list of {len(v)} {type(v[0]).__name__}')
    else:
        print(f'  {k:20s}: {v}')

## 5. CMDFusion Model & Mixed Precision (AMP)

In [ ]:
import sys, importlib
if 'model_cmdfusion_uav' in sys.modules:
    importlib.reload(sys.modules['model_cmdfusion_uav'])
from model_cmdfusion_uav import CMDFusionUAV, build_cmdfusion_uav

model_config = {
    'num_classes': NUM_CLASSES,
    'input_dims': INPUT_DIMS,
    'hiden_size': HIDEN_SIZE,
    'scale_list': SCALE_LIST,
    'min_volume_space': MIN_VOLUME_SPACE,
    'max_volume_space': MAX_VOLUME_SPACE,
    'spatial_shape': SPATIAL_SHAPE,
    'img_target_size': IMG_TARGET_SIZE,
    'backbone_2d': BACKBONE_2D,
    'pretrained_2d': PRETRAINED_2D,
    'lambda_seg2d': LAMBDA_SEG2D,
    'lambda_xm': LAMBDA_XM,
    'ignore_label': 0,
    'seg_labelweights': class_weights.tolist(),
}

model = build_cmdfusion_uav(model_config).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CMDFusion -- Total params: {total_params:,}  |  Trainable: {trainable:,}')

# Optimizer, scheduler, and scaler
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(f'Optimizer: Adam (lr={LR})')
print(f'Scheduler: StepLR (step={LR_STEP}, gamma={LR_DECAY})')
print(f'Loss: CMDFusion internal (CE + Lovasz + KD MSE)')
print(f'AMP FP16 Acceleration: {USE_AMP}')

## 6. Training Loop

In [ ]:
def move_to_device(batch, device):
    """Move batch tensors to device, handling mixed types in data_dict."""
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(device, non_blocking=True)
        elif isinstance(v, list) and len(v) > 0 and isinstance(v[0], torch.Tensor):
            batch[k] = [t.to(device, non_blocking=True) for t in v]
    return batch


def train_one_epoch(model, loader, optimizer, scaler, device, use_amp=USE_AMP):
    model.train()
    total_loss = 0.0; total_correct = 0; total_seen = 0
    for batch_idx, batch in enumerate(loader):
        batch = move_to_device(batch, device)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            data_dict = model(batch)
            loss = data_dict['loss']

        if torch.isnan(loss) or torch.isinf(loss):
            print(f'    [WARNING] NaN/Inf loss at batch {batch_idx}, skipping')
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        if GRAD_CLIP > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        # Use fused 3D logits as primary output
        if 'fuse_pts_scale_all' in data_dict:
            preds = data_dict['fuse_pts_scale_all'].argmax(1)
        else:
            preds = data_dict['logits'].argmax(1)
        targets = data_dict['labels']

        n_pts = len(targets)
        total_loss += loss.item() * n_pts
        valid_mask = targets != IGNORE_INDEX
        total_correct += (preds[valid_mask] == targets[valid_mask]).sum().item()
        total_seen += valid_mask.sum().item()

        if (batch_idx + 1) % 50 == 0:
            print(f'    batch {batch_idx+1}/{len(loader)}  loss={loss.item():.4f}')

    return total_loss / max(total_seen, 1), total_correct / max(total_seen, 1)


@torch.no_grad()
def validate(model, loader, device, num_classes=NUM_CLASSES, use_amp=USE_AMP):
    model.eval()
    total_loss = 0.0; all_preds = []; all_targets = []
    for batch in loader:
        batch = move_to_device(batch, device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            data_dict = model(batch)
            loss = data_dict['loss']

        if 'fuse_pts_scale_all' in data_dict:
            preds = data_dict['fuse_pts_scale_all'].argmax(1)
        else:
            preds = data_dict['logits'].argmax(1)

        n_pts = len(data_dict['labels'])
        total_loss += loss.item() * n_pts
        all_preds.append(preds.cpu().numpy().flatten())
        all_targets.append(data_dict['labels'].cpu().numpy().flatten())

    metrics = compute_metrics(
        np.concatenate(all_preds), np.concatenate(all_targets), num_classes)
    avg_loss = total_loss / max(sum(len(p) for p in all_preds), 1)
    return avg_loss, metrics

In [ ]:
if CKPT_PATH is not None:
    if not os.path.exists(CKPT_PATH):
        raise FileNotFoundError(
            f'CKPT_PATH={CKPT_PATH!r} does not exist. '
            'Refusing to silently evaluate a random model.'
        )
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    if 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'Loaded from epoch {ckpt["epoch"]} (val mIoU={ckpt["miou"]:.4f})')
    else:
        model.load_state_dict(ckpt)
        print(f'Loaded weights from: {CKPT_PATH}')
    history = None
else:
    print(f'Starting training: {MAX_EPOCH} epochs, validate every {VAL_EVERY} epochs')
    print('=' * 70)
    best_miou = -1.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_miou': []}
    for epoch in range(1, MAX_EPOCH + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scaler, DEVICE, USE_AMP)
        scheduler.step()
        elapsed = time.time() - t0
        batches_per_sec = len(train_loader) / max(elapsed, 1e-4)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        print(f'Epoch {epoch:3d}/{MAX_EPOCH} | '
              f'train_loss={train_loss:.4f}  acc={train_acc:.4f} | '
              f'lr={scheduler.get_last_lr()[0]:.6f} | {elapsed:.1f}s ({batches_per_sec:.1f} batches/s)')

        if epoch % VAL_EVERY == 0 or epoch == MAX_EPOCH:
            val_loss, val_m = validate(
                model, val_loader, DEVICE, NUM_CLASSES, USE_AMP)
            history['val_loss'].append(val_loss)
            history['val_miou'].append(val_m['miou'])
            print(f'  >> VAL  loss={val_loss:.4f}  '
                  f'acc={val_m["accuracy"]:.4f}  '
                  f'mIoU (fg 18 classes)={val_m["miou"]:.4f}  '
                  f'n_ignored={val_m["n_ignored"]}')
            for i, name in enumerate(CLASS_NAMES):
                tag = ' (Background - excluded from mIoU)' if i == 0 else ''
                print(f'     [{i:2d}] {name:20s} IoU = {val_m["per_class_iou"][i]:.4f}{tag}')
            if val_m['miou'] > best_miou:
                best_miou = val_m['miou']
                save_path = os.path.join(SAVE_DIR, 'cmdfusion_best.pth')
                torch.save({'epoch': epoch, 'miou': best_miou,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'config': model_config},
                           save_path)
                print(f'  >> Saved best model (mIoU={best_miou:.4f}) -> {save_path}')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print('Training complete.')

## 6b. Training Curves

In [ ]:
if history is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(range(1, MAX_EPOCH+1), history['train_loss'], 'b-', label='Train')
    if history['val_loss']:
        val_e = [i*VAL_EVERY for i in range(1, len(history['val_loss'])+1)]
        if MAX_EPOCH % VAL_EVERY != 0: val_e[-1] = MAX_EPOCH
        axes[0].plot(val_e, history['val_loss'], 'ro-', label='Val')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss', fontsize=13); axes[0].legend()
    axes[1].plot(range(1, MAX_EPOCH+1), history['train_acc'], 'b-', label='Train')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training Accuracy', fontsize=13); axes[1].legend()
    if history['val_miou']:
        axes[2].plot(val_e, history['val_miou'], 'go-', label='Val mIoU (fg 18 classes)')
        axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('mIoU')
        axes[2].set_title('Validation mIoU (Foreground)', fontsize=13); axes[2].legend()
    plt.suptitle('CMDFusion Training Curves', fontsize=15, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Training curves not available -- loaded from checkpoint.')

## 7. Test Evaluation

Evaluates the best CMDFusion checkpoint on the test split.
Reports overall accuracy, mIoU (foreground 18 classes), and per-class IoU.

In [ ]:
# ============================================================
# 1. Load trained best checkpoint
# ============================================================
load_path = CKPT_PATH if CKPT_PATH else os.path.join(SAVE_DIR, 'cmdfusion_best.pth')
if not os.path.exists(load_path):
    raise FileNotFoundError(f'No checkpoint at {load_path}. Train first or set CKPT_PATH.')
ckpt = torch.load(load_path, map_location=DEVICE, weights_only=False)
if 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded model from epoch {ckpt["epoch"]} (val mIoU={ckpt["miou"]:.4f})')
else:
    model.load_state_dict(ckpt)
    print(f'Loaded weights from: {load_path}')

# ============================================================
# Test Evaluation
# ============================================================
print('\n>> Running Test Evaluation...')
test_loss, test_m = validate(model, test_loader, DEVICE, NUM_CLASSES, USE_AMP)

print('=' * 60)
print('TEST RESULTS (18 foreground classes; Background excluded from mIoU)')
print('=' * 60)
print(f'Overall Accuracy: {test_m["accuracy"]:.4f}')
print(f'Mean IoU (mIoU):  {test_m["miou"]:.4f}')
print(f'Test Loss:        {test_loss:.4f}')
print('-' * 60)
for i, name in enumerate(CLASS_NAMES):
    tag = ' (Background - excluded from mIoU)' if i == 0 else ''
    print(f'  [{i:2d}] {name:20s} IoU = {test_m["per_class_iou"][i]:.4f}{tag}')
print('=' * 60)

## 8. Confusion Matrix

In [ ]:
cm = test_m['confusion_matrix']
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_title('Test Confusion Matrix (19 classes; Background = class 0)', fontsize=13)
thresh = cm.max() / 2.0
for r in range(NUM_CLASSES):
    for c in range(NUM_CLASSES):
        ax.text(c, r, f'{cm[r, c]:,}', ha='center', va='center',
                color='white' if cm[r, c] > thresh else 'black', fontsize=7)
plt.tight_layout(); plt.show()

## 9. Per-Class IoU Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(CLASS_NAMES, test_m['per_class_iou'],
              color=[PALETTE[i] for i in range(NUM_CLASSES)],
              edgecolor='black', linewidth=0.8)
ax.axhline(test_m['miou'], color='red', linestyle='--',
           label=f'mIoU (fg 18 classes)={test_m["miou"]:.4f}')
ax.set_ylim(0, 1.05)
ax.set_ylabel('IoU', fontsize=12)
ax.set_title('Per-Class IoU -- Test Set (19 classes; Background excluded from mIoU)', fontsize=13)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
ax.legend(fontsize=11)
for bar, iou in zip(bars, test_m['per_class_iou']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{iou:.3f}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

## 10. Qualitative Predictions (BEV)

Visualizes a test frame with:
- Ground Truth labels in BEV
- CMDFusion predictions in BEV
- Camera FOV coverage

In [ ]:
# ---- Qualitative results: GT vs CMDFusion Prediction ----
model.eval()
test_frame_idx = min(50, len(test_ds) - 1)

# Get a single test sample through the dataset
sample = test_ds[test_frame_idx]
lidar_path = sample['root']
seq_name = test_ds.frames[test_frame_idx][3]

# Load full raw points for BEV visualization
pts_raw = np.load(lidar_path).astype(np.float64)
pts_full = pts_raw[:, :3]
labels_26 = np.load(test_ds.frames[test_frame_idx][2]).astype(np.int64)
labels_full = map_labels_26_to_19(labels_26)

# Create a single-sample batch and run inference
single_batch = collate_fn_uav([sample])
single_batch = move_to_device(single_batch, DEVICE)

with torch.no_grad(), torch.cuda.amp.autocast(enabled=USE_AMP):
    out = model(single_batch)

if 'fuse_pts_scale_all' in out:
    pred_labels = out['fuse_pts_scale_all'].argmax(1).cpu().numpy()
else:
    pred_labels = out['logits'].argmax(1).cpu().numpy()

# The predictions are for volume-filtered points only
mask = sample['mask']
pred_full = np.full(len(pts_full), 0, dtype=np.int64)  # default to background
pred_full[mask] = pred_labels

bev_full = lidar_xyz_to_bev_coords(pts_full)

# Camera FOV mask
calib = get_calibration(seq_name)
u, v, valid_front = project_lidar_to_image(pts_full, calib)
with Image.open(test_ds.frames[test_frame_idx][1]) as img_pil:
    img_w, img_h = img_pil.size
valid_fov = valid_front & (u >= 0) & (u < img_w) & (v >= 0) & (v < img_h)

def color_for(labels):
    c = np.zeros((len(labels), 3))
    ignore_mask = labels == IGNORE_INDEX
    c[~ignore_mask] = PALETTE[labels[~ignore_mask]]
    c[ignore_mask] = IGNORE_COLOR
    return c

valid_mask_eval = labels_full != IGNORE_INDEX
acc = (pred_full[valid_mask_eval] == labels_full[valid_mask_eval]).mean() * 100 if valid_mask_eval.any() else float('nan')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].scatter(bev_full[:, 0], bev_full[:, 1], c=color_for(labels_full), s=0.8, alpha=0.8, linewidths=0)
axes[0].set_title(f'Ground Truth (BEV, {len(pts_full):,} raw pts)', fontsize=13)
axes[0].set_aspect('equal'); axes[0].axis('off')

axes[1].scatter(bev_full[:, 0], bev_full[:, 1], c=color_for(pred_full), s=0.8, alpha=0.8, linewidths=0)
axes[1].set_title(f'CMDFusion Prediction (acc={acc:.1f}%)', fontsize=13)
axes[1].set_aspect('equal'); axes[1].axis('off')

axes[2].scatter(bev_full[valid_fov, 0], bev_full[valid_fov, 1], c='tab:green', s=0.8, alpha=0.8, label=f'In FOV ({valid_fov.sum():,})', linewidths=0)
axes[2].scatter(bev_full[~valid_fov, 0], bev_full[~valid_fov, 1], c='tab:red', s=0.8, alpha=0.8, label=f'Outside FOV ({(~valid_fov).sum():,})', linewidths=0)
axes[2].set_title('Camera FOV Coverage', fontsize=13)
axes[2].set_aspect('equal'); axes[2].axis('off')
axes[2].legend(fontsize=9, loc='upper right')

legend_elements = [Patch(facecolor=PALETTE[i], label=f'{i}: {CLASS_NAMES[i]}') for i in range(NUM_CLASSES)]
fig.legend(handles=legend_elements, loc='lower center', ncol=7, fontsize=8)
plt.suptitle(f'{seq_name} -- CMDFusion Prediction ({len(pts_full):,} points)', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()